In [1]:
import numpy as np
import pandas as pd

In [2]:
def filter_by_missing(dataframe, threshold=0.3):
    """
    Filter a dataframe to keep only columns with missing values percentage below the threshold.
    
    Parameters:
    -----------
    dataframe : pandas.DataFrame
        The input dataframe to filter
    threshold : float, default=0.3
        The maximum acceptable percentage of missing values (0.0 to 1.0)
        
    Returns:
    --------
    pandas.DataFrame
        Filtered dataframe with only columns that have missing values below the threshold
    """
    # Calculate percentage of missing values for each column
    missing_percentage = dataframe.isna().mean()
    keep_columns = missing_percentage[missing_percentage < threshold].index
    return dataframe[keep_columns]

### Import

In [3]:
df_bmx = pd.read_csv("OUTPUT\\df_bmx.csv")
df_bpxo = pd.read_csv("OUTPUT\\df_bpxo.csv")
df_demo = pd.read_csv("OUTPUT\\df_demo.csv")
df_lab = pd.read_csv("OUTPUT\\df_lab.csv")
survey_dy = pd.read_csv("OUTPUT\\survey_dy.csv")
survey_jh = pd.read_csv("OUTPUT\\survey_jh.csv")

In [4]:
columns_to_drop = [
    'RIAGENDR', 'RIDAGEYR', 'DMDEDUC2', 'ALQ121', 'ALQ130', 'BPQ020', 
    'BPQ030', 'BPQ050A', 'BPQ080', 'BPQ100D', 'DED120', 'DED125', 
    'DIQ010', 'DIQ050', 'DIQ070', 'DBQ700', 'DBD895', 'DBD900', 
    'DBD905', 'HUQ010', 'HUQ090', 'INDFMMPI', 'LBXSGTSI', 'LBXMCVSI',
]

survey_jh = survey_jh.drop(columns=[col for col in columns_to_drop if col in survey_jh.columns])

if 'SEQN' in survey_jh.columns:
    survey_jh = survey_jh.rename(columns={'SEQN': 'respondent_id'})

if 'respondent_id.1' in df_bpxo.columns:
    df_bpxo.drop('respondent_id.1', axis=1, inplace=True)

### Join

In [5]:
for df in [df_bmx, df_bpxo, df_demo, df_lab, survey_dy, survey_jh]:
   if 'Unnamed: 0' in df.columns:
       df.drop(columns=['Unnamed: 0'], inplace=True)

df_bmx = df_bmx.drop_duplicates(subset=['respondent_id'])
df_bpxo = df_bpxo.drop_duplicates(subset=['respondent_id'])
df_demo = df_demo.drop_duplicates(subset=['respondent_id'])
df_lab = df_lab.drop_duplicates(subset=['respondent_id'])
survey_dy = survey_dy.drop_duplicates(subset=['respondent_id'])
survey_jh = survey_jh.drop_duplicates(subset=['respondent_id'])

merged_data = df_demo.merge(df_bmx, on='respondent_id', how='left', suffixes=('', '_bmx'))
merged_data = merged_data.merge(df_bpxo, on='respondent_id', how='left', suffixes=('', '_bpxo'))
merged_data = merged_data.merge(df_lab, on='respondent_id', how='left', suffixes=('', '_lab'))
merged_data = merged_data.merge(survey_dy, on='respondent_id', how='left', suffixes=('', '_dy'))
merged_data = merged_data.merge(survey_jh, on='respondent_id', how='left', suffixes=('', '_jh'))

# Drop any additional duplicate columns that might remain
duplicate_cols = [col for col in merged_data.columns if col.endswith(('_bmx', '_bpxo', '_lab', '_dy', '_jh')) and col.split('_')[0] in merged_data.columns]
merged_data.drop(columns=duplicate_cols, errors='ignore', inplace=True)

# Remove duplicate column if it appears twice
if 'current_aspirin_use' in merged_data.columns and 'current_aspirin_use_bmx' in merged_data.columns:
    merged_data.drop(columns=['current_aspirin_use_bmx'], inplace=True)

duplicate_cols = [col for col in merged_data.columns if 
                 any(col.endswith(suffix) for suffix in ['_bmx', '_bpxo', '_lab', '_dy', '_jh'])]
merged_data.drop(columns=duplicate_cols, errors='ignore', inplace=True)

merged_data_filtered = filter_by_missing(merged_data, threshold=0.3)

merged_data_filtered.to_csv("OUTPUT\\df_dataset.csv", index=False)